# Подготовка агрегированных данных о поведении пользователей платформы интернет торговли NorthCart с целью обучения модели ML

- Автор: Федоров Алексей Анатольевич
- Дата: 21.10.2026

<a id='цели-и-задачи-проекта'></a>
## Цели и задачи проекта
Торговая онлайн платформа NorthCart использует ML для разбиения клиентов на сегменты, показывающие вероятность покупики в ближайшие 7 дней. Затем, использует эти данные для обучения модели и прогнозирования покупок. Для этого используются два типа признаков:
- **Онлайн-признаки** — данные, доступные сразу в момент запроса. Чаще всего это сырые данные о действиях пользователей на сайте. Например, указанный пользователем способ оплаты, устройство пользователя или данные о последней сессии.
  
- **Батч-признаки** — агрегированные данные о поведении пользователей за последние дни и недели. Это некоторые показатели, которые формируются из онлайн-признаков по расписанию и сохраняются в хранилище. Например, количество просмотров товаров за последние 30 дней, число сессий за 7 дней или сумма заказов за месяц.

Батч-признаки формируются на основе длительной истории событий, поэтому рассчитывать их онлайн на постоянной основе слишком дорого. Чтобы их создать, нужно загружать большой объём данных о пользовательской истории, объединять несколько источников и пересчитывать окна агрегации, что увеличивает задержку ответа.
Поэтому, требуется реализовать DAG, который будет регулярно запускаться и расчитывать батч-признаки на определённую дату run_date. 

**1.** **Тип задачи** - подготовка агрегированных данных за период времени < run_date при помощи DAG Airflow в пакетном и ручном режимах на выбор пользователя.

**2.** **Задача** - реализовать DAG, который при разовом запуске относительно logical_date (context.get("ds")) строит два временных среза с агрегацией признаков для customer_id. Перечень создаваемых признаков будет дан ниже. В результирующем датасете должна быть одна строка на каждую пару customer_id, run_date. Поскольку, для каждой logical_date DAG запускается отдельно, то и датасеты будут раздельными по датам, и дату хранить в виде колонки не требуется. DAG должен уметь обуслуживать как ручные запуски так и через планировщик AirFlow. Для ручного запуска должна быть возможноть передачи даты среза на UI Airflow через переменную batch_features_run_date, а для других сценариев и командной строки нужно использовать logical_date из контекста Airflow.

**3.** **Цель** - ускорить обучение ML-модели и предсказание значений, чтобы она могла использовать уже готовые данные, а не расчитывать их сама.

**4.** **Целевая переменная** - не будет расчитываться на исторических данных. Это вынесено в отдельную задачу.

**5.** **Ограничения**  
**5.1.** DAG запускается раз в день `0 0 * * *`, нужо обеспечить независимость Time Zone от настроек сервера, так как в airflow.config TZ может быть изменена с UTC на другую. Временные метки данных в таблицах PostgreSql заданы как timestamp и signup_date как date. То есть, без TZ, это **naivy** формат. Поскольку, описанием не определено, в какой TZ даты, то будем считать, что в UTC.

**5.2.** Дата для построения среза run_date берётся из Variable **batch_features_run_date**, а если не задано, берётся **logical_date** DAG, а если DAG запущен вручную в Airflow 3 без параметров, то logical_date будет None и мы останавливаем DAG с ошибкой. Это позволяет корректно работать по ручному вызову и в пакетном режиме через backfill и по расписанию и из командной строки.

**5.3** Результат работы при каждом запуске DAG представляет из себя датасет с одной строкой на каждый customer_id. TARGET не считается, поэтому для нас нет разницы, исторические данные или инференс. Поэтому, всегда будем выгружать в S3 повторно-воспроизводимый партиционированный parket `s3://<bucket>/batch_features/run_date=2025-09-01/features.parquet`.

**6. Режимы запуска DAG**

| № | Режим запуска | Как передаётся дата | Откуда берётся `run_date` |
|---:|---|---|---|
| 1 | Вручную в UI, Variable задана | Admin → Variables: `batch_features_run_date = 2025-09-01`, затем Trigger DAG | Variable: `2025-09-01` |
| 2 | Вручную в UI, Variable не задана, дата указана | Trigger DAG → поле Logical date: `2025-10-01` | `logical_date`: `2025-10-01` |
| 3 | Вручную в UI, Airflow 3, без даты и без Variable | Trigger DAG без параметров | `logical_date = None`, DAG падает с ошибкой «logical_date не задан» |
| 4 | Вручную в UI, Airflow 2, без даты и без Variable | Trigger DAG без параметров | `logical_date = now()`, срез строится на сегодня (ошибки не будет, см. примечание 1) |
| 5 | CLI, через Variable | `airflow variables set batch_features_run_date 2025-09-01`<br>`airflow dags trigger batch_features` | Variable: `2025-09-01` |
| 6 | CLI, через `logical_date` | Airflow 2: `airflow dags trigger batch_features -e 2025-10-01`<br>Airflow 3: `airflow dags trigger batch_features --logical-date 2025-10-01` | `logical_date`: `2025-10-01` |
| 7 | Backfill (пакетный пересчёт) | Airflow 2: `airflow dags backfill batch_features -s 2025-09-01 -e 2025-10-01`<br>Airflow 3: `airflow backfill create --dag-id batch_features --from-date 2025-09-01 --to-date 2025-10-01` | `logical_date` каждого запуска: свой срез на каждую дату (Variable должна быть удалена, см. примечания 2 и 3) |
| 8 | По расписанию | `schedule="@daily"`: Airflow сам создаёт запуски | `logical_date` интервала, например `2025-10-01` (Variable должна быть удалена, примечания 2 и 3) |



<a id='описание-данных'></a>
## Описание данных
### Сырые данные в базе для агрегации
#### Таблица `customers` (17 846 строк)

Справочник уникальных пользователей.

- `customer_id` — уникальный идентификатор пользователя. Базовый ключ, по которому строятся все батч-признаки.
- `signup_date` — дата регистрации пользователя. На этапе предобработки нужно привести к единому типу времени.
- Также хранится базовая информация: страна проживания, возраст и другие признаки (не все нужны для расчёта).

#### Таблица `sessions` (37 699 строк, 16 894 уникальных пользователя)

Данные о сессиях и контексте визитов на сайт.

- `session_id` — уникальный идентификатор сессии. Нужен для связи с таблицей `events` и удаления дубликатов сессий.
- `customer_id` — идентификатор пользователя, запустившего сессию. Нужен для агрегации метрик на уровне клиента.
- `start_time` — время начала сессии. Используется для фильтрации по `run_date`, формирования срезов и расчёта длины сессий.

#### Таблица `events` (239 089 строк, 37 699 уникальных сессий)

Логи действий пользователей внутри системы (воронка покупки).

- `event_id` — уникальный идентификатор события. Нужен для удаления дубликатов в логах.
- `session_id` — идентификатор сессии. Через него событие связывается с пользователем (в самой таблице `events` нет `customer_id`).
- `timestamp` — время события. Нужно для фильтрации по `run_date` и расчёта признаков по окнам.
- `event_type` — тип события:
  - `page_view` — просмотр страницы;
  - `add_to_cart` — добавление товара в корзину;
  - `checkout` — процедура оплаты;
  - `purchase` — совершённая покупка.
- `product_id` — идентификатор товара. Нужен для расчёта числа уникальных товаров. Для товарных признаков используются только строки, где `product_id` задан.

#### Таблица `orders` (10 652 строки, 8 230 уникальных покупателей)

Данные о фактах покупок (используется для денежных признаков).

- `order_id` — уникальный идентификатор заказа. Для подсчёта числа заказов и удаления дубликатов.
- `customer_id` — идентификатор пользователя, оформившего заказ.
- `order_time` — время оформления заказа. Для фильтрации по `run_date`, расчёта окна 30 дней и давности последней покупки.
- `total_usd` — итоговая сумма заказа. Для расчёта суммарной выручки и среднего чека.

### Столбцы - результаты агрегации сырых данных

- T — run_date 00:00 в DATA_TZ (UTC). Граница не включается: все данные берутся строго < T.
- w — окно, 7 или 30 дней. Окно — это интервал `[T − w дней, T)`.
- E_w — события в окне после предобработки: timestamp ∈ `[T−w, T)`, без дублей по event_id, customer_id взят через JOIN sessions.
- S_w — сессии с start_time ∈ `[T−w, T)`.
- O_w — заказы с order_time ∈ `[T−w, T)`.
- safe_div(a, b) — a / b, а если b = 0, то 0.
- Каркас таблицы — все customer_id из customers. У клиентов без активности счётчики равны 0.

| № | Признак | Источник | Формула | Если данных нет |
|---:|---|---|---|---:|
| 1 | `customer_id` | `customers` | Одна строка на клиента | — |
| 2 | `run_date` | Параметр | `YYYY-MM-DD` | — |
| 3–4 | `page_view_cnt_7d`, `_30d` | `events` | `count(E_w, page_view)` | `0` |
| 5–6 | `add_to_cart_cnt_7d`, `_30d` | `events` | `count(E_w, add_to_cart)` | `0` |
| 7–8 | `view_to_cart_conv_7d`, `_30d` | `events` | `safe_div(add_to_cart, page_view)` | `0` |
| 9–10 | `cart_to_purchase_conv_7d`, `_30d` | `events` | `safe_div(purchase, add_to_cart)` | `0` |
| 11–12 | `unique_products_7d`, `_30d` | `events`, `product_id IS NOT NULL` | `nunique(product_id)` | `0` |
| 13 | `avg_session_duration_30d` | `sessions` + `events` | `mean(max(ts) − min(ts))` по `S_30`, в секундах | `0` |
| 14–15 | `sessions_cnt_7d`, `_30d` | `sessions` | `count(S_w)` | `0` |
| 16 | `days_since_last_purchase` | `orders`, вся история `< T` | `floor((T − max(order_time)) / 1 день)` | `-1` |
| 17 | `orders_cnt_30d` | `orders` | `count(O_30)` | `0` |
| 18 | `revenue_30d` | `orders` | `sum(total_usd)` | `0.0` |
| 19 | `avg_order_value_30d` | `orders` | `mean(total_usd)` по `O_30` | `0.0` |

##  Импорт библиотек, выполнение базовых настроек

### Обновление pip

In [1]:
%pip install --upgrade --user --disable-pip-version-check -i https://mirrors.tuna.tsinghua.edu.cn/pypi/web/simple pip

Looking in indexes: https://mirrors.tuna.tsinghua.edu.cn/pypi/web/simple
    matplotlib-inline (<0.2.0appnope,>=0.1.0) ; platform_system == "Darwin"
                      ~~~~~~~~^
Note: you may need to restart the kernel to use updated packages.


In [2]:
# .local/lib/python3.9/site-packages должен стоять до глобального окружения
import sys
for p in sys.path:
    print(p)

/home/jovyan/work
/opt/conda/lib/python39.zip
/opt/conda/lib/python3.9
/opt/conda/lib/python3.9/lib-dynload

/home/jovyan/.local/lib/python3.9/site-packages
/opt/conda/lib/python3.9/site-packages
/opt/conda/lib/python3.9/site-packages/IPython/extensions
/home/jovyan/.ipython


In [3]:
%pip -V

pip 26.0.1 from /home/jovyan/.local/lib/python3.9/site-packages/pip (python 3.9)
Note: you may need to restart the kernel to use updated packages.


### Проверка совместимости версий модулей и наличия wheels

In [4]:
%pip install --dry-run --only-binary=:all: --disable-pip-version-check \
    numpy==1.26.4 pandas==2.1.4 "sqlalchemy==1.4.54" psycopg2-binary==2.9.10 \
    pendulum==3.0.0 boto3==1.35.99 "urllib3<1.27"

Note: you may need to restart the kernel to use updated packages.


### Установка модулей
Пакеты ставятся не в базовое окружение conda (/opt/conda/lib/python3.9/site-packages), а в домашний каталог пользователя: `~/.local/lib/python3.9/site-packages`

При импорте Python ищет модули в этом каталоге раньше, чем в системном. Поэтому наша версия пакета перекрывает системную, а сама системная остаётся на месте.

Версии подобраны под Python **3.9**, что стоит вместе с Jupyter. Для Airflow потребуется подгонка, там **3.12**.

In [5]:
%pip install --user --only-binary=:all: --disable-pip-version-check --no-warn-script-location \
    numpy==1.26.4 \
    scipy==1.13.1 \
    pandas==2.1.4 \
    sqlalchemy==1.4.54 \
    psycopg2-binary==2.9.10 \
    pendulum==3.0.0 \
    boto3==1.35.99 \
    "urllib3<1.27" \
    matplotlib==3.8.2 \
    seaborn==0.13.0 \
    "tqdm>=4.66" \
    ipywidgets==8.1.5

    matplotlib-inline (<0.2.0appnope,>=0.1.0) ; platform_system == "Darwin"
                      ~~~~~~~~^
Note: you may need to restart the kernel to use updated packages.


👉 Тут **Нужен Restart Kernel !!!**

#### Делаем импорты и проверяем что встало

In [6]:
import sys
import warnings
from getpass import getpass
from dataclasses import dataclass, field, fields
from IPython.display import Markdown

from typing_extensions import Protocol
from typing import Optional, Dict, List

import numpy as np
import pandas as pd
import pendulum
import sqlalchemy
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
import boto3

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

In [7]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)
warnings.filterwarnings("ignore", category=FutureWarning)

PG_HOST = 'rc1d-pdq3kahro9enkfir.mdb.yandexcloud.net'
PG_PORT = 6432
PG_DB = 'playground_ds_20261015_a868c33dd7'
PG_USER = 'ds_20261015_a868c33dd7'
PG_PASSWORD = '7be84cc613cf45548f876a6e4602662a'  

DB_SCHEMA_NAME = 'public'

DATA_TZ = 'UTC' 

In [8]:
print("python    ", sys.version.split()[0], '\n')

for name, mod in [("numpy", np), ("pandas", pd), ("sqlalchemy", sqlalchemy),
                  ("pendulum", pendulum), ("boto3", boto3)]:
    print(f"{name:10s} {mod.__version__}")

python     3.9.5 

numpy      1.26.4
pandas     2.1.4
sqlalchemy 1.4.54
pendulum   3.0.0
boto3      1.35.99


<a id='общие-функции'></a>
## Общие функции
<a id='базовые'></a>
### Базовые

<a id='базовый-eda'></a>
### Базовый EDA

In [9]:
def get_missings(df, hide_zeros = False):
    missing_df = df.isna().sum().to_frame(name='missing_count')
    missing_df['missing_percent'] = (df.isna().mean() * 100).round(2)
    if hide_zeros:
        missing_df = missing_df.query('missing_percent > 0')
    
    return missing_df.sort_values(by='missing_percent', ascending=False)

def describe_date_column(series: pd.Series) -> pd.Series:
    return pd.Series({
        'count': series.count(),
        'missing': series.isna().sum(),
        'min': series.min(),
        'max': series.max(),
        'range_days': (series.max() - series.min()).days,
        'unique_periods': series.nunique(),
        'most_common': series.mode().iloc[0] if not series.mode().empty else None,
    })

def get_reference_info(df, name, numeric=False, limit = 50):
    """
        Выполняет проверку серии name из DataFrame df как категории. А именно:
            - выводит TOP 50 уникальных значений серии и общее число уникальных значений;
            - выводит счётчик встречаемости и % для каждого значения в серии name;
            - если серия числовая, то ещё выводит min и max значения.
        Таким образом, помогает оценить качество справочника или просто серии (колонки) и заметить неявные дубликаты.
    """
    values = df[name].drop_duplicates().sort_values()
    len_values = len(values)
    lst = values.head(limit).to_list()
    
    elipsis = ', ...' if len_values > limit else ''
    display(Markdown(f"**{name}:** `{lst}{elipsis}` = **{len_values}**\n\n"))
        
    counts_df = df[name].value_counts(dropna=False).astype('float64').to_frame('count')
    counts_df.index = counts_df.index.map(lambda x: '<NA>' if pd.isna(x) else x)
    counts_df['%'] = (counts_df['count'] / counts_df['count'].sum() * 100).round(2)        
    counts_df['count'] = counts_df['count'].astype('Int64')
             
    display(counts_df)
            
    print('\nПропущено:', df[name].isna().sum())
    
    if numeric:
        values_num = pd.to_numeric(df[name], errors='coerce')        
        min_val = values_num.min()
        max_val = values_num.max()
        print(f'min = {min_val}, mean = {round(values_num.mean(), 2)}, max = {max_val}, median = {values_num.median()}')
                
        whisker_left, whisker_right = get_whiskers(values_num, left=min_val, right=max_val)
        print(f'left_whisker = {whisker_left}, right_whisker = {whisker_right}')

        non_numeric = values[values_num.isna()].tolist()
        if non_numeric:
            print(f'Нецифровые значения: {non_numeric}')
    print('\n')

##  Подключение к БД

In [10]:
engine = create_engine(URL.create(
    drivername="postgresql+psycopg2",
    username=PG_USER, password=PG_PASSWORD,
    host=PG_HOST, port=PG_PORT, database=PG_DB,
))

assert sqlalchemy.__version__.startswith("1.4"), "Нужна SQLAlchemy 1.4 (как в Airflow 2.10)"
assert pd.__version__.startswith("2.1"), "Нужен pandas 2.1.x (как в образе Airflow)"

with engine.connect() as conn:
    result = conn.execute(text("SELECT current_database(), current_user, version()"))        
    rows = result.mappings().all()        
    display(pd.DataFrame(rows))

,current_database,current_user,version
0,playground_ds_20261015_a868c33dd7,ds_20261015_a868c33dd7,PostgreSQL 17.10 (Ubuntu 17.10-201-yandex.6009...


# **Часть 1. Знакомство с данными в PostgreSQL**


#### **Задание 1.1: загрузите и изучите данные**


## Загрузка данных из базы

In [11]:
TIMESTAMP = "timestamp"   # Postgres timestamp → datetime64[ns, UTC], точность до мкс
DATE = "date"             # Postgres date      → datetime64[ns, UTC], время 00:00
DATE_KINDS = (TIMESTAMP, DATE)
ID_TYPE = "Int64"

class TableProto(Protocol):
    name: str
    description: str
    columns: Dict[str, str]
    df: Optional[pd.DataFrame]    


class NorthCartTablesProto(Protocol):
    customers: TableProto
    orders: TableProto
    sessions: TableProto
    events: TableProto
        
@dataclass
class Table(TableProto):
    name: str
    description: str
    columns: Dict[str, str]
    df: Optional[pd.DataFrame] = None

    @property
    def dtype(self) -> Dict[str, str]:
        return {c: t for c, t in self.columns.items() if t not in DATE_KINDS}

    @property
    def parse_dates(self) -> Dict[str, dict]:
        return {c: {"utc": True} for c, t in self.columns.items() if t in DATE_KINDS}

    @property
    def date_columns(self) -> List[str]:
        return [c for c, t in self.columns.items() if t == DATE]

    def select_sql(self, schema_name: str) -> str:
        cols = ", ".join(f'"{c}"' for c in self.columns)
        return f'SELECT {cols} FROM "{schema_name}"."{self.name}"'

    def postprocess(self) -> None:
        """Приводит DATE-столбцы к точности до дня."""
        for c in self.date_columns:
            s = self.df[c]
            not_midnight = (s.notna() & (s != s.dt.normalize())).sum()
            if not_midnight:
                print(f"⚠️ {self.name}.{c}: {not_midnight} значений не на полночь, обнуляю время")
            self.df[c] = s.dt.normalize()  
            
@dataclass
class NorthCartTables(NorthCartTablesProto):
    customers: Table = field(default_factory=lambda: Table(
        name="customers", description="Клиенты",
        columns={
            "customer_id": ID_TYPE,           # int4 NOT NULL, PK
            "signup_date": DATE,         # date
        },
    ))
    orders: Table = field(default_factory=lambda: Table(
        name="orders", description="Покупки клиентов",
        columns={
            "order_id": ID_TYPE,              # int4 NOT NULL, PK
            "customer_id": ID_TYPE,           # int4 NULL
            "order_time": TIMESTAMP,     # timestamp
            "total_usd": "float64",      # float8
        },
    ))
    sessions: Table = field(default_factory=lambda: Table(
        name="sessions", description="Сессии",
        columns={
            "session_id": ID_TYPE,            # int4 NOT NULL, PK
            "customer_id": ID_TYPE,           # int4 NULL
            "start_time": TIMESTAMP,     # timestamp
        },
    ))
    events: Table = field(default_factory=lambda: Table(
        name="events", description="События",
        columns={
            "event_id": ID_TYPE,              # int4 NULL, без PK
            "session_id": ID_TYPE,            # int4 NULL
            "timestamp": TIMESTAMP,      # timestamp
            "event_type": "category",    # varchar(50)
            "product_id": ID_TYPE,            # float8 → целое, как products.product_id (int4)
        },
    ))

In [12]:
def load_data(conn, schema_name) -> NorthCartTables:
    data = NorthCartTables()
    
    for f in fields(data):
        table_object = getattr(data, f.name) 
        table_name = table_object.name
        table_object.df = pd.read_sql(
            sql=text(table_object.select_sql(schema_name)),
            con=conn,
            dtype=table_object.dtype,
            parse_dates=table_object.parse_dates,
        )
        table_object.postprocess()
    
        # помечаем датасет
        table_object.df.attrs['name'] = table_name    
    
        df_shape = table_object.df.shape
        print(f'✅ Таблица {table_name} загружена, содержит: {df_shape[0]} строк x {df_shape[1]} колонок')    
    
    return data

def pre_analyze_ds(df, id_name):
    n = len(df)

    print(f'{id_name} уникален: {df[id_name].is_unique}')
    display(df.dtypes.to_frame(name='type'))
    display(df.head())

    body_cols = [col for col in df.columns if col != id_name]

    full_dup = df.duplicated().sum()
    body_dup = df.duplicated(subset=body_cols).sum()

    print(f'Явные полные дубликаты: {full_dup} ({full_dup / n:.2%})')
    print(f'Дубликаты тела без {id_name}: {body_dup} ({body_dup / n:.2%})')

    display(get_missings(df))

In [13]:
data = None
with engine.connect() as conn:
    data = load_data(conn, DB_SCHEMA_NAME)

✅ Таблица customers загружена, содержит: 17846 строк x 2 колонок
✅ Таблица orders загружена, содержит: 10652 строк x 4 колонок
✅ Таблица sessions загружена, содержит: 37699 строк x 3 колонок
✅ Таблица events загружена, содержит: 239089 строк x 5 колонок


👉 Добавить загрузку в DAG.

## Базовый EDA

### Клиент

In [14]:
data.customers.df['customer_id'].nunique()

17846

In [15]:
pre_analyze_ds(data.customers.df, 'customer_id')

customer_id уникален: True


,type
customer_id,Int64
signup_date,"datetime64[ns, UTC]"


,customer_id,signup_date
0,1,2020-09-04 00:00:00+00:00
1,2,2020-04-05 00:00:00+00:00
2,3,2023-08-31 00:00:00+00:00
3,4,2022-06-30 00:00:00+00:00
4,5,2022-07-22 00:00:00+00:00


Явные полные дубликаты: 0 (0.00%)
Дубликаты тела без customer_id: 15715 (88.06%)


,missing_count,missing_percent
customer_id,0,0.0
signup_date,0,0.0


In [16]:
describe_date_column(data.customers.df.signup_date)

count                                 17846
missing                                   0
min               2020-01-01 00:00:00+00:00
max               2025-10-31 00:00:00+00:00
range_days                             2130
unique_periods                         2131
most_common       2024-07-19 00:00:00+00:00
dtype: object

#### Промежуточный вывод

Поля соотвествуют описанию, дубликатов нет (проверка без customer_id не считается), пропусков нет, id уникален, данные доступны за 5 лет: 2020 - 2025, последний год неполный, заканчивается на октябре. Всего дано 2130 дней.

### Покупки клиента

In [17]:
data.orders.df['order_id'].nunique()

10652

In [18]:
pre_analyze_ds(data.orders.df, 'order_id')

order_id уникален: True


,type
order_id,Int64
customer_id,Int64
order_time,"datetime64[ns, UTC]"
total_usd,float64


,order_id,customer_id,order_time,total_usd
0,1,13917,2025-01-31 23:07:42+00:00,85.72
1,2,1022,2024-02-19 01:17:50+00:00,116.17
2,3,6145,2024-12-04 20:24:13+00:00,137.35
3,4,3152,2024-07-17 08:50:47+00:00,27.35
4,11,14044,2025-03-12 02:48:39+00:00,47.40


Явные полные дубликаты: 0 (0.00%)
Дубликаты тела без order_id: 0 (0.00%)


,missing_count,missing_percent
order_id,0,0.0
customer_id,0,0.0
order_time,0,0.0
total_usd,0,0.0


In [19]:
describe_date_column(data.orders.df.order_time)

count                                 10652
missing                                   0
min               2024-01-01 07:08:06+00:00
max               2025-10-31 22:59:41+00:00
range_days                              669
unique_periods                        10652
most_common       2024-01-01 07:08:06+00:00
dtype: object

In [20]:
# проверка связи orders --> customers
lost_orders = data.orders.df.loc[
    ~data.orders.df['customer_id'].isin(data.customers.df['customer_id']),
    'customer_id'].unique()
print(f'Ордера без customer: {len(lost_orders)}')

Ордера без customer: 0


In [21]:
customers_with_no_orders = data.customers.df.loc[
    ~data.customers.df['customer_id'].isin(data.orders.df['customer_id']),
    'customer_id'].unique()
print(f'Клиенты без ордеров: {len(customers_with_no_orders)}')

Клиенты без ордеров: 9616


In [22]:
# проверка числа покупок по клиентам среди тех, кто делал не менее 1-ой покупки
op = data.orders.df.groupby('customer_id').size()
display(op.describe())

count    8230.000000
mean        1.294289
std         0.562233
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         5.000000
dtype: float64

In [23]:
counts = op.value_counts().sort_index()
shares = op.value_counts(normalize=True).sort_index() * 100

pd.DataFrame({
    "count": counts,
    "share_%": shares.round(2),
    "cum_%": shares.cumsum().round(2),
})

,count,share_%,cum_%
1,6203,75.37,75.37
2,1681,20.43,95.80
3,299,3.63,99.43
4,45,0.55,99.98
5,2,0.02,100.00


#### Промежуточный вывод

Поля соотвествуют описанию, дубликатов нет, id уникален, пропусков нет, данные о покупках доступны примерно за 2 года: 2024 - 2025 октябрь, или 669 дней. Последний год неполный. Потерянный ордеров нет, но есть клиенты, у которых нет покупок: 9616 клиентов из 17846 (53.9%). То есть, примерно каждый второй делал покупки за доступные 2 года.

Из тех клиентов, кто делал покупки, 75% имеют только 1 покупку, а 20% 2 покупки. Максимальное число - 5 покупок на клиента.

### Сессии клиента

In [24]:
data.sessions.df['session_id'].nunique()

37699

In [25]:
pre_analyze_ds(data.sessions.df, 'session_id')

session_id уникален: True


,type
session_id,Int64
customer_id,Int64
start_time,"datetime64[ns, UTC]"


,session_id,customer_id,start_time
0,2,13917,2025-01-31 21:29:42+00:00
1,3,1022,2024-02-19 00:52:50+00:00
2,4,2882,2024-08-04 19:54:31+00:00
3,9,6476,2024-02-22 16:57:44+00:00
4,10,6145,2024-12-04 18:53:13+00:00


Явные полные дубликаты: 0 (0.00%)
Дубликаты тела без session_id: 0 (0.00%)


,missing_count,missing_percent
session_id,0,0.0
customer_id,0,0.0
start_time,0,0.0


In [26]:
describe_date_column(data.sessions.df.start_time)

count                                 37699
missing                                   0
min               2024-01-01 00:57:40+00:00
max               2025-10-31 23:34:11+00:00
range_days                              669
unique_periods                        37686
most_common       2024-01-09 13:04:33+00:00
dtype: object

In [27]:
# проверка связи sessions --> customers
lost_sessions = data.sessions.df.loc[
    ~data.sessions.df['customer_id'].isin(data.customers.df['customer_id']),
    'customer_id'].unique()
print(f'Сессии без customer: {len(lost_sessions)}')

Сессии без customer: 0


In [28]:
# проверка числа сессий по клиентам
op = data.sessions.df.groupby('customer_id').size()
display(op.describe())

count    16894.000000
mean         2.231502
std          1.216600
min          1.000000
25%          1.000000
50%          2.000000
75%          3.000000
max         10.000000
dtype: float64

In [29]:
counts = op.value_counts().sort_index()
shares = op.value_counts(normalize=True).sort_index() * 100

pd.DataFrame({
    "count": counts,
    "share_%": shares.round(2),
    "cum_%": shares.cumsum().round(2),
})

,count,share_%,cum_%
1,5647,33.43,33.43
2,5381,31.85,65.28
3,3383,20.02,85.30
4,1607,9.51,94.81
5,634,3.75,98.57
6,174,1.03,99.60
7,48,0.28,99.88
8,18,0.11,99.99
9,1,0.01,99.99
10,1,0.01,100.00


In [30]:
lost_orders = data.orders.df.loc[
    ~data.orders.df['customer_id'].isin(data.sessions.df['customer_id']),
    'order_id'].unique()
print(f'Ордера без session: {len(lost_orders)}')

Ордера без session: 0


#### Промежуточный вывод

Поля соотвествуют описанию, дубликатов нет, id уникален, пропусков нет, данные о сессиях доступны за примерно 2 года 2024 - 2025 отябрь, или 669 дней, что соотвествует таблице orders. Потерянных сессий нет. Ордеров без сессий тоже нет.

Большинство клиентов (28%), из тех, что имеют сессии, имеют 1-3 сессии. Максимальное число сессий - 10, но это всего у одного клиента.

### События

In [31]:
pre_analyze_ds(data.events.df, 'event_id')

event_id уникален: True


,type
event_id,Int64
session_id,Int64
timestamp,"datetime64[ns, UTC]"
event_type,category
product_id,Int64


,event_id,session_id,timestamp,event_type,product_id
0,11,2,2025-01-31 21:48:42+00:00,page_view,929
1,12,2,2025-01-31 21:57:42+00:00,page_view,226
2,13,2,2025-01-31 21:59:07+00:00,add_to_cart,226
3,14,2,2025-01-31 22:01:42+00:00,page_view,864
4,15,2,2025-01-31 22:25:42+00:00,page_view,1151


Явные полные дубликаты: 0 (0.00%)
Дубликаты тела без event_id: 0 (0.00%)


,missing_count,missing_percent
product_id,24881,10.41
event_id,0,0.00
session_id,0,0.00
timestamp,0,0.00
event_type,0,0.00


In [32]:
body = ["session_id", "timestamp", "event_type", "product_id"]
print(f"events: подозрительных повторов тела: {data.events.df.duplicated(subset=body).sum()}")

events: подозрительных повторов тела: 0


👉 Добавить в DAG.

In [33]:
# проверим, в каких типах события находятся пропуски product_id
(data.events.df["product_id"].isna()
   .groupby(data.events.df["event_type"], observed=True)
   .agg(missing="sum", share="mean"))

,missing,share
event_type,,
add_to_cart,0,0.0
checkout,14229,1.0
page_view,0,0.0
purchase,10652,1.0


👉 Пропуски product_id не случайны. Это структурный детерменированный пропуск: продукт есть только у события add и view, а у корзины (события checkout и purchase) его нет. В корзине может быть много продуктов.

In [34]:
describe_date_column(data.events.df.timestamp)

count                                239089
missing                                   0
min               2024-01-01 01:02:40+00:00
max               2025-11-01 01:50:04+00:00
range_days                              670
unique_periods                       238548
most_common       2024-12-27 16:17:33+00:00
dtype: object

In [35]:
get_reference_info(data.events.df, 'event_type')

**event_type:** `['add_to_cart', 'checkout', 'page_view', 'purchase']` = **4**



,count,%
event_type,,
page_view,169300,70.81
add_to_cart,44908,18.78
checkout,14229,5.95
purchase,10652,4.46



Пропущено: 0




In [36]:
# проверка связи events --> sessions
lost_events = ~data.events.df["session_id"].isin(data.sessions.df["session_id"])
print(f"События без сессии: {lost_events.sum()}")

События без сессии: 0


In [37]:
# проверка числа событий по сессиям
op = data.events.df.groupby('session_id').size()
display(op.describe())

count    37699.000000
mean         6.342052
std          3.429223
min          1.000000
25%          3.000000
50%          6.000000
75%          9.000000
max         17.000000
dtype: float64

In [38]:
counts = op.value_counts().sort_index()
shares = op.value_counts(normalize=True).sort_index() * 100

pd.DataFrame({
    "count": counts,
    "share_%": shares.round(2),
    "cum_%": shares.cumsum().round(2),
})

,count,share_%,cum_%
1,3457,9.17,9.17
2,3169,8.41,17.58
3,2901,7.70,25.27
4,3215,8.53,33.80
5,3278,8.70,42.49
6,3375,8.95,51.45
7,3566,9.46,60.91
8,3532,9.37,70.28
9,3294,8.74,79.01
10,2911,7.72,86.73


#### Промежуточный вывод

Поля соотвествуют описанию, дубликатов нет, id уникален, пропуски есть только у product_id (10.4%), но они не случайны, чётко определены типом события - view и add. Данные о событиях доступны примерно за 2 года 2024 - 2025 ноябряь, или 670 дней, что соотвествует таблице sessions с учётом длительности последней сессии. Потерянных сессий нет.

По типу событий прослеживается сильно выраженная воронка: 70% событий просмотры и только 4.46% покупки. 8.48% сессий имеют 1 - 11 событий, в данном диапазоне распределение почти равномерное. Максимальное число - 17 событий только у 2-х сессий.

Число записей:
- customers: 17 846;
- sessions: 37 699;
- orders: 10 652.

# **Часть 2. Подготовка batch-признаков**


#### **Задание 2.1: подготовьте данные**

#### **Задание 2.2: рассчитайте признаки**


## Обработка данных

Правильные типы pandas уже обеспечены при загрузке из Postgre за счёт схемы данных: проекция + парсинг (dtypes). Правильность дат так же обеспечена схемой данных при загрузке за счёт parse_dates. 
По результатам базового EDA и по заданию нужно добавить:
- защиту от дубликатов в events;
- удаление строк, где пропуск в первичных или внешних ключах. Это возможно в некоторых таблицах исходя из схемы в Postgre, и такие строки бесполезны в работе.

✅ На случай, если дубликаты по event_id появятся, в DAG будем использовать удаление с сохранением первого.

In [39]:
def dedup_events(df: pd.DataFrame) -> pd.DataFrame:
    before = len(df)
    # в Postgre `event_id int4 NULL`, хотя это странно
    null_id = df["event_id"].isna()
    
    deduped = (df[~null_id]
               .sort_values(["event_id", "timestamp"])
               .drop_duplicates(subset="event_id", keep="first"))
    df = pd.concat([deduped, df[null_id]]).sort_index()

    removed = before - len(df)
    print(f"events: удалено дублей по event_id: {removed} ({removed / before:.2%})")

    # только диагностика: одинаковое тело события под разными event_id
    body = ["session_id", "timestamp", "event_type", "product_id"]
    print(f"events: подозрительных повторов тела: {df.duplicated(subset=body).sum()}")
    return df

👉 Добавить в DAG.

✅ Удаление бесполезных строк.

In [40]:
KEYS = {
    "customers": ["customer_id"],
    "orders":    ["order_id", "customer_id"],
    "sessions":  ["session_id", "customer_id"],
    "events":    ["session_id"],   
}

def drop_null_keys(table: Table) -> None:
    cols = KEYS[table.name]
    mask = table.df[cols].isna().any(axis=1)
    if mask.any():
        print(f"⚠️ {table.name}: удалено строк с NULL в {cols}: {mask.sum()}")
        table.df = table.df[~mask]

In [41]:
for f in fields(data):
    table_object = getattr(data, f.name) 
    drop_null_keys(table_object)

👉 Добавить в DAG.

## Агрегация обработанных данных в витрину

✅ Фиксация полей результирующей витрины.

In [42]:
FEATURE_COLUMNS = [
    "customer_id", "run_date",
    "page_view_cnt_7d", "page_view_cnt_30d",
    "add_to_cart_cnt_7d", "add_to_cart_cnt_30d",
    "view_to_cart_conv_7d", "view_to_cart_conv_30d",
    "cart_to_purchase_conv_7d", "cart_to_purchase_conv_30d",
    "unique_products_7d", "unique_products_30d",
    "avg_session_duration_30d",
    "sessions_cnt_7d", "sessions_cnt_30d",
    "days_since_last_purchase",
    "orders_cnt_30d", "revenue_30d", "avg_order_value_30d",
]

✅ Подготовка функции create_features, создающей витрину из сырых, исправленных по EDA  данных.

In [43]:
WINDOWS = (7, 30)

def safe_div(num, den):
    """num / den, при den <= 0 возвращает 0.0."""
    return num.div(den.where(den > 0)).fillna(0.0).astype("float64")

def create_features(run_date, data: NorthCartTables) -> pd.DataFrame:
    T = pd.Timestamp(run_date, tz=DATA_TZ)
    
    def win_start(w: int) -> pd.Timestamp:
        return T - pd.Timedelta(days = w)        
    
    # клиентов фильтруем по signup_date, чтобы были раньше run_date, так же, оставляем клиентов с неизвестной 
    # signup_date, чтобы они не пропали
    cust = data.customers.df[~(data.customers.df.signup_date >= T)]
    ses = data.sessions.df[data.sessions.df.start_time < T]
    ordr = data.orders.df[data.orders.df.order_time < T]
    ev = data.events.df[data.events.df.timestamp < T]
    
    # --- 1. каркас ---
    base = cust[['customer_id']].drop_duplicates().set_index('customer_id')
    # --- 2. добавляем дату для надёжности (дата по всему пакету одинаковая) ---
    base["run_date"] = T.strftime("%Y-%m-%d")
      
    # добавляем customer_id в events из sessions, чтобы группировать
    ev = ev.merge(data.sessions.df[['session_id', 'customer_id']], on='session_id', how='inner')
    
    # --- 3-4, 5-6. ---
    # считаем page_view_cnt_7d, _30d, add_to_cart_cnt_7d, _30d и промежуточный purchase_cnt 
    ev_feats = []
    ses_feats = []
    for w in WINDOWS:
        start_time = win_start(w)
        e = ev[ev.timestamp >= start_time]
        
        # группируем по событиям, создаём только строки для тех, что были у customer_id
        # переворачиваем строки в колонки с заполнением пропусков по каждому customer_id
        # задаём нужные нам колонки и заполняем попуски
        counts = (e.groupby(['customer_id', 'event_type'], observed=True).size()
                    .unstack(fill_value = 0)
                    .reindex(columns=['page_view', 'add_to_cart', 'purchase'], fill_value=0))
        # переименовываем колонки, чтобы было так, как нам нужно
        counts.columns = [f'{c}_cnt_{w}d' for c in counts.columns]
       
        # --- 11-12. unique_products_7d, _30d ---
        products = (e[e.product_id.notna()]
                      .groupby('customer_id')['product_id'].nunique().rename(f'unique_products_{w}d'))
                  
        ev_feats += [counts, products]
    
        # --- 14-15. sessions_cnt_7d, _30d ---     
        session_count = ses[ses.start_time >= start_time].groupby('customer_id').size().rename(f'sessions_cnt_{w}d')
        ses_feats += [session_count]
        
    # --- 13. avg_session_duration_30d ---
    s30 = ses[ses.start_time >= win_start(30)][['session_id', 'customer_id']]

    duration = (ev[ev.session_id.isin(s30.session_id)]
       .groupby('session_id')['timestamp']
       .agg(['min', 'max']))
    duration = (duration['max'] - duration['min']).dt.total_seconds().rename('dur')
    s30 = s30.merge(duration, left_on='session_id', right_index=True, how='left')
    s30['dur'] = s30['dur'].fillna(0.0)
    
    avg_dur = s30.groupby('customer_id')['dur'].mean().rename('avg_session_duration_30d')
    
    # --- 17, 18, 19. orders_cnt_30d, revenue_30d, avg_order_value_30d ---
    o30 = ordr[ordr.order_time >= win_start(30)]
    ord_feats = o30.groupby('customer_id').agg(
        orders_cnt_30d = ('order_id', 'nunique'),
        revenue_30d = ('total_usd', 'sum'),          
        avg_order_value_30d = ('total_usd', 'mean'), 
    )
    
    # --- 16. days_since_last_purchase ---
    last = ordr.groupby('customer_id')['order_time'].max()
    days_last = (T - last).dt.days.rename('days_since_last_purchase')   # .days округляет вниз
    
    # собираем витрину
    features = base.join(ev_feats + ses_feats + [avg_dur, ord_feats, days_last], how='left')
    
    # после left join могут появиться пропуски, поэтому, заполняем их
    cnt_cols = [c for c in features.columns if "_cnt_" in c or c.startswith("unique_products")]
    features[cnt_cols] = features[cnt_cols].fillna(0).astype("int64")
    
    money_cols = ["avg_session_duration_30d", "revenue_30d", "avg_order_value_30d"]
    features[money_cols] = features[money_cols].fillna(0.0).astype("float64")
    
    features["days_since_last_purchase"] = features["days_since_last_purchase"].fillna(-1).astype("int64")
    
    # --- 7,8,9,10. view_to_cart_conv_7d, _30d, cart_to_purchase_conv_7d, _30d ---
    for w in WINDOWS:
        features[f"view_to_cart_conv_{w}d"] = \
            safe_div(features[f"add_to_cart_cnt_{w}d"], features[f"page_view_cnt_{w}d"])
            
        features[f"cart_to_purchase_conv_{w}d"] = \
            safe_div(features[f"purchase_cnt_{w}d"], features[f"add_to_cart_cnt_{w}d"])
    
    features = features.reset_index()
    features = features[FEATURE_COLUMNS]
    
    return features

## Проверка витрины

In [44]:
# RUN_DATE = '2025-09-01'
RUN_DATE = '2025-10-09'
features = create_features(RUN_DATE, data)

In [45]:
assert features["customer_id"].is_unique

T = pd.Timestamp(RUN_DATE, tz=DATA_TZ)
cust = data.customers.df[~(data.customers.df.signup_date >= T)]
base = cust[['customer_id']].drop_duplicates().set_index('customer_id')

assert len(features) == len(base)
assert features.isna().sum().sum() == 0
assert (features["sessions_cnt_7d"] <= features["sessions_cnt_30d"]).all()
assert (features["page_view_cnt_7d"] <= features["page_view_cnt_30d"]).all()

In [46]:
features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17641 entries, 0 to 17640
Data columns (total 19 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   customer_id                17641 non-null  Int64  
 1   run_date                   17641 non-null  object 
 2   page_view_cnt_7d           17641 non-null  int64  
 3   page_view_cnt_30d          17641 non-null  int64  
 4   add_to_cart_cnt_7d         17641 non-null  int64  
 5   add_to_cart_cnt_30d        17641 non-null  int64  
 6   view_to_cart_conv_7d       17641 non-null  float64
 7   view_to_cart_conv_30d      17641 non-null  float64
 8   cart_to_purchase_conv_7d   17641 non-null  float64
 9   cart_to_purchase_conv_30d  17641 non-null  float64
 10  unique_products_7d         17641 non-null  int64  
 11  unique_products_30d        17641 non-null  int64  
 12  avg_session_duration_30d   17641 non-null  float64
 13  sessions_cnt_7d            17641 non-null  int

In [47]:
display(features.describe().T)

,count,mean,std,min,25%,50%,75%,max
customer_id,17641.0,10010.576895,5787.372633,1.0,4987.0,10029.0,15016.0,19999.0
page_view_cnt_7d,17641.0,0.09597,0.74011,0.0,0.0,0.0,0.0,14.0
page_view_cnt_30d,17641.0,0.451618,1.619589,0.0,0.0,0.0,0.0,22.0
add_to_cart_cnt_7d,17641.0,0.025395,0.24265,0.0,0.0,0.0,0.0,6.0
add_to_cart_cnt_30d,17641.0,0.123803,0.530456,0.0,0.0,0.0,0.0,7.0
view_to_cart_conv_7d,17641.0,0.005696,0.055231,0.0,0.0,0.0,0.0,1.0
view_to_cart_conv_30d,17641.0,0.026208,0.113118,0.0,0.0,0.0,0.0,1.0
cart_to_purchase_conv_7d,17641.0,0.004438,0.060718,0.0,0.0,0.0,0.0,1.0
cart_to_purchase_conv_30d,17641.0,0.020752,0.130124,0.0,0.0,0.0,0.0,1.0
unique_products_7d,17641.0,0.095176,0.732591,0.0,0.0,0.0,0.0,14.0


In [48]:
T = pd.Timestamp(RUN_DATE, tz="UTC")
S30 = T - pd.Timedelta(days=30)
ids = features["customer_id"]

# сессии
s = data.sessions.df
raw = s[(s.start_time >= S30) & (s.start_time < T) & s.customer_id.isin(ids)]
assert len(raw) == features.sessions_cnt_30d.sum()

# заказы и выручка
o = data.orders.df
raw_o = o[(o.order_time >= S30) & (o.order_time < T) & o.customer_id.isin(ids)]
assert len(raw_o) == features.orders_cnt_30d.sum()
assert np.isclose(raw_o.total_usd.sum(), features.revenue_30d.sum())

# 7d не больше 30d
for p in ["page_view_cnt", "add_to_cart_cnt", "unique_products", "sessions_cnt"]:
    assert (features[f"{p}_7d"] <= features[f"{p}_30d"]).all(), p

# конверсии в [0, 1]: в этих данных так и есть
conv = features.filter(like="_conv_")
assert conv.ge(0).all().all()
print("max conv:", conv.max().to_dict())

max conv: {'view_to_cart_conv_7d': 1.0, 'view_to_cart_conv_30d': 1.0, 'cart_to_purchase_conv_7d': 1.0, 'cart_to_purchase_conv_30d': 1.0}


✅ Найдём самый активный период по числу покупок

In [49]:
o = data.orders.df

# заказы и выручка по дням UTC (пустые дни = 0)
daily = (o.set_index("order_time")
           .resample("D")
           .agg(orders=("order_id", "size"), revenue=("total_usd", "sum")))

def by_run_date(w: int) -> pd.DataFrame:
    """Сумма за окно [T − w, T) для каждого возможного T."""
    r = daily.rolling(w).sum().dropna()          # на дату d: дни d−w+1 … d
    r.index = r.index + pd.Timedelta(days=1)     # T = d + 1, правая граница не включается
    return r

r30 = by_run_date(30)
r7 = by_run_date(7)

# T в пределах данных: полная 30-дневная история и не позже конца выгрузки
r30 = r30[r30.index <= pd.Timestamp("2025-11-01", tz="UTC")]

cand = r30.join(r7, rsuffix="_7d").sort_values(["orders", "orders_7d"], ascending=False)
display(cand.head(10))

best_T = cand.index[0]
run_date = best_T.strftime("%Y-%m-%d")
print("run_date:", run_date, "| заказов за 30d:", int(cand.iloc[0]["orders"]),
      "| за 7d:", int(cand.iloc[0]["orders_7d"]))

,orders,revenue,orders_7d,revenue_7d
order_time,,,,
2025-10-09 00:00:00+00:00,528.0,71323.50,113.0,14920.25
2024-08-09 00:00:00+00:00,526.0,69817.99,128.0,16941.16
2025-10-15 00:00:00+00:00,523.0,69257.45,110.0,13734.45
2024-08-10 00:00:00+00:00,522.0,70198.39,130.0,17713.96
2025-10-08 00:00:00+00:00,522.0,70100.89,119.0,16524.73
2025-10-10 00:00:00+00:00,522.0,70904.44,115.0,15320.71
2025-10-13 00:00:00+00:00,521.0,69742.84,118.0,15642.60
2025-10-14 00:00:00+00:00,521.0,69362.75,112.0,14456.74
2024-08-08 00:00:00+00:00,520.0,69371.37,132.0,19960.79


run_date: 2025-10-09 | заказов за 30d: 528 | за 7d: 113


👉 дата для тестирования **2025-10-09**

## Вывод

**Витрина признаков.** 

Функция **create_features(run_date, data)** строит витрину из 19 столбцов: customer_id, run_date и 17 признаков на основе агрегации. Каждая группа признаков считается отдельной агрегацией, затем все группы присоединяются к базе клиентов через LEFT JOIN. Окна заданы как `[T − w, T)`, всё время в UTC.


Срез на 2025-10-09. Эта дата выбрана, потому что за 30 дней до неё было больше всего заказов: 528 на сумму $71.3K.


В тестовой витрине 17 641 клиент: 205 клиентов зарегистрировались после даты среза, поэтому они исключены.
Пропусков нет. Нулевое значение признака означает, что активности не было. Значение −1 в days_since_last_purchase означает, что покупок до T не было.
Активных клиентов немного: около `10%` за 30 дней и около `2%` за 7 дней. Поэтому у большинства признаков медиана равна 0.
Конверсии лежат в диапазоне [0, 1]. Значения за 7 дней не больше значений за 30 дней.
Число сессий, число заказов и выручка за 30 дней по витрине совпадают с подсчётом напрямую по исходным таблицам.

**О выборе тестовой даты среза**. 

Пик пологий. Лучшие даты отличаются меньше чем на `2%`: от 520 до 528 заказов за 30 дней. Рядом идут периоды в октябре 2025 и в августе 2024. Значит, 2025-10-09 — типичный активный срез, а не всплеск от разовой акции.